In [16]:
import uuid, time
from pyspark.sql.functions import col, lit
from lakehouse_engine.config import get_notebook_parameters, get_root_path, SilverConfig
from lakehouse_engine.spark import get_spark_session
from lakehouse_engine.utils import setup_logging
from lakehouse_engine.io import DataFrameReader, DataFrameWriter

In [17]:
#PROVISORIO SPARK LOCAL
import ipywidgets as widgets
from IPython.display import display

dataset_widget = widgets.Text(value='customers', description='Dataset:')

display(dataset_widget)

Text(value='customers', description='Dataset:')

In [18]:
layer = "silver"

#Load Spark Session and get parameters Databricks/Local
spark = get_spark_session()
dataset = get_notebook_parameters(dataset_widget)

# Load logging
log = setup_logging(f"{dataset}_{layer}")

silver_cfg = SilverConfig(dataset, layer)
silver_cfg.load_yaml()

In [19]:
# Create variables from yaml
target_table = silver_cfg.target_table
target_schema = silver_cfg.target_schema
target_catalog = silver_cfg.target_catalog
source_table = silver_cfg.source_table
source_schema = silver_cfg.source_schema
source_catalog = silver_cfg.source_catalog

# Create logical variables
run_id = str(uuid.uuid4())

# Create paths
root_path = get_root_path()
file_path = f"{root_path}/{target_catalog}/landing/files/{dataset}/"
checkpoint_path = f"{root_path}/{target_catalog}/landing/checkpoints/{dataset}/"

In [20]:
log.info(f"[PARAMS] dataset={dataset} catalog={target_catalog} layer={layer} run_id={run_id}")
log.info(f"[CTX] target_table={target_table}")
log.info(f"[CTX] file_path={file_path}")
log.info(f"[CTX] checkpoint_path={checkpoint_path}")
start = time.time()
log.info(f"[SILVER][START] run_id={run_id}")

try:
    reader = DataFrameReader(spark)
    writer = DataFrameWriter(spark)
    
    # Read table
    df_bronze = reader.read_table(
        catalog=source_catalog, 
        schema=source_schema, 
        table=source_table
    )
    count_bronze = df_bronze.count()
    log.info(f"[STEP 1] READ: Bronze table loaded: {count_bronze} rows")
    
    # Apply schema
    df_schema = silver_cfg.apply_schema(df_bronze)
    log.info(f"[STEP 2] SCHEMA: Applied schema")

    # Deduplicate
    df_dedup = silver_cfg.deduplicate(df_schema)
    log.info(f"[STEP 3] DEDUP: Deduplicated by {silver_cfg.primary_keys}")
    
    # Add metadata columns (_processed_ts, _run_id, keep _ingest_ts)
    df_final = silver_cfg.add_metadata(df_dedup, run_id)
    log.info(f"[STEP 4] METADATA: Added _processed_ts and _run_id columns")
    
    # Write table - UPSERT
    writer.upsert_table(df_final, f"{target_schema}.{target_table}", silver_cfg.primary_keys)
    count_final = df_final.count()
    log.info(f"[STEP 5] WRITE: Delta table merged (UPSERT): {target_schema}.{target_table} with {count_final} rows")
    
    duration_s = round(time.time() - start, 3)
    log.info(f"[SILVER][SUCCESS] run_id={run_id} dataset={dataset} total_rows={count_final} duration_s={duration_s}s")

except Exception as e:
    duration_s = round(time.time() - start, 3)
    log.error(f"[SILVER][FAILED] run_id={run_id} duration_s={duration_s} error={repr(e)}")
    raise